# Label Forge - Google Colab Training Notebook
This notebook automatically trains YOLO models on Google Colab GPU

In [ ]:
# @title Setup Parameters
JOB_ID = "job_id_here"  # @param {type:"string"}
DATASET_URL = "https://backend.com/datasets/xyz.zip"  # @param {type:"string"}
CALLBACK_URL = "https://backend.com/training/job_id/colab-callback"  # @param {type:"string"}

print(f"Job ID: {JOB_ID}")
print(f"Dataset URL: {DATASET_URL}")
print(f"Callback URL: {CALLBACK_URL}")

In [ ]:
# Install dependencies
!pip install -q ultralytics torch torchvision requests gdown

In [ ]:
# Check GPU
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# Download dataset
import gdown
import zipfile
import os

os.makedirs("dataset", exist_ok=True)

print("[DOWNLOAD] Fetching dataset...")
gdown.download(DATASET_URL, "dataset.zip", quiet=False)

print("[EXTRACT] Extracting dataset...")
with zipfile.ZipFile("dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("dataset")
print("[EXTRACT] Done!")

In [ ]:
# Train YOLO model
from ultralytics import YOLO
import torch

print("[TRAIN] Starting YOLO training...")

model = YOLO('yolov8n.pt')
results = model.train(
    data='dataset/data.yaml',
    epochs=50,
    imgsz=640,
    patience=5,
    device=0 if torch.cuda.is_available() else 'cpu',
    save=True,
    verbose=True,
)

print("[TRAIN] Training complete!")

In [ ]:
# Extract metrics
metrics = {
    "map_score": float(results.box.map50 if hasattr(results.box, 'map50') else 0),
    "precision": float(results.box.mp if hasattr(results.box, 'mp') else 0),
    "recall": float(results.box.mr if hasattr(results.box, 'mr') else 0),
}

print("[METRICS]")
for key, value in metrics.items():
    print(f"  {key}: {value:.4f}")

In [ ]:
# Send callback to backend
import requests
import json
from datetime import datetime

print("[CALLBACK] Sending results to backend...")

callback_data = {
    "status": "done",
    "metrics": metrics,
    "model_url": "https://drive.google.com/file/d/model_id" if False else None,  # Optional: upload to cloud
    "finished_at": datetime.utcnow().isoformat()
}

try:
    response = requests.post(CALLBACK_URL, json=callback_data, timeout=30)
    print(f"[CALLBACK] Response: {response.status_code}")
    print(f"[CALLBACK] {response.text}")
except Exception as e:
    print(f"[ERROR] Callback failed: {str(e)}")

## Training Complete! ✅
Your model has been trained and the results have been sent back to Label Forge.